# 分布式训练：工业界的标准工具链

> 一张卡，训得动前面那些小模型，训不动真正的大模型。
>
> 多卡训练是工业界的默认设置。这一节学着用三个标准工具：Accelerate、ZeRO、Megatron-LM。

前面几节写的代码都是单进程的：一个模型、一个 optimizer、一个 for 循环。真实世界里，7B 模型的 AdamW 全量训练光固定开销就要 112 GB 显存，单张 A100 80 GB 连参数带优化器都装不下。

工业界的答案不是「自己手写分布式通信」，而是站在现成工具上。这一节换一种讲法：不逐行实现通信原语，而是把你在真实项目里一定会碰到的三样东西讲透：

- **Accelerate**：HuggingFace 的统一启动层，一份训练脚本切换 DDP / FSDP / DeepSpeed；
- **ZeRO 参数**：显存不够时真正会去调的那几个配置项（stage、offload、overlap_comm……）；
- **Megatron-LM**：从零预训练百亿级以上模型的重型武器，3D 并行的工业标准。

一条主线贯穿始终：**微调用 HF 生态（Accelerate + ZeRO / FSDP），从零预训练大模型用 Megatron**。

## 1. 先算一笔账：单卡为什么不够

先回答「为什么要分布式」。AdamW + 混合精度训练下，每个参数占 16 bytes：FP16 参数 2 bytes、FP16 梯度 2 bytes、FP32 master 权重 4 bytes、AdamW 一阶矩 4 bytes、二阶矩 4 bytes。这部分是和序列长度无关的固定开销，7B 参数就是 112 GB——还没算激活值。

多卡最朴素的用法是数据并行（Distributed Data Parallel，DDP）：每张卡持有一份完整模型副本，把数据切成 N 份分给 N 张卡，各自前向反向，再用 all-reduce 把梯度平均。它的吞吐接近线性增长，但有个致命问题——**显存一点没省**：8 张卡总共 896 GB 显存里，有 7/8 是一模一样的冗余副本。

In [ ]:
# === 单卡显存账单：7B 模型 ===
P = 7e9  # 7B 参数

param_fp16   = 2 * P   # FP16 参数
grad_fp16    = 2 * P   # FP16 梯度
master_fp32  = 4 * P   # FP32 master 权重
adam_m       = 4 * P   # AdamW 一阶矩
adam_v       = 4 * P   # AdamW 二阶矩

fixed_bytes = param_fp16 + grad_fp16 + master_fp32 + adam_m + adam_v
fixed_gb = fixed_bytes / 1e9

print(f"模型参数 (FP16):    {param_fp16 / 1e9:.1f} GB")
print(f"梯度 (FP16):        {grad_fp16 / 1e9:.1f} GB")
print(f"master 权重 (FP32): {master_fp32 / 1e9:.1f} GB")
print(f"AdamW m (FP32):     {adam_m / 1e9:.1f} GB")
print(f"AdamW v (FP32):     {adam_v / 1e9:.1f} GB")
print(f"固定开销小计:       {fixed_gb:.0f} GB (= 16 × P bytes)")
print()
print("关键观察：16P = 2P 参数 + 2P 梯度 + 12P 优化器状态。")
print("DDP 下这三块在每张卡上都是完整副本——冗余就在这里。")

## 2. ZeRO：把冗余切到多卡

ZeRO（Zero Redundancy Optimizer）是微软 2020 年提出的方法，也是今天所有主流训练框架显存优化的基石。思路一句话：既然每张卡上的 16P bytes 都是冗余副本，那就按三块逐层切到 N 张卡上，切完的卡互相配合，逻辑上仍然等价于一个完整的大模型。

三个 Stage 递进，每升一级多切一块：

- **Stage 1**：切优化器状态（12P → 12P/N），每卡只更新自己那 1/N 参数的 optimizer；
- **Stage 2**：再切梯度（2P → 2P/N），反向时用 reduce-scatter 让每卡只留自己那片梯度；
- **Stage 3**：再切参数（2P → 2P/N），前向反向用到哪层就临时 all-gather 哪层，用完即丢。

In [ ]:
# === ZeRO 三 Stage 显存手算：7B 模型 × 8 卡 ===
P = 7e9
N = 8

print(f"{'方案':<16}{'参数':>8}{'梯度':>8}{'优化器':>8}{'单卡合计':>10}{'相对 DDP':>10}")
print("-" * 64)
configs = [
    ("DDP",          2 * P,       2 * P,       12 * P),
    ("ZeRO Stage 1", 2 * P,       2 * P,       12 * P / N),
    ("ZeRO Stage 2", 2 * P,       2 * P / N,   12 * P / N),
    ("ZeRO Stage 3", 2 * P / N,   2 * P / N,   12 * P / N),
]
ddp_bytes = 16 * P
for name, p, g, o in configs:
    total = p + g + o
    print(f"{name:<16}{p/1e9:>7.1f} {g/1e9:>7.1f} {o/1e9:>7.1f} "
          f"{total/1e9:>8.1f} GB {total/ddp_bytes*100:>8.1f}%")
print()
print("关键观察：从 DDP 到 Stage 3，单卡显存 112 GB → 14 GB，压缩到 1/8。")
print("代价是通信量递增：Stage 越高，前向反向要传递的碎片越多。")
print()
print("工业经验：Stage 2 是性价比最高的档位（通信开销 ≈ DDP）；")
print("参数实在放不下才上 Stage 3（对节点间带宽更敏感）。")

## 3. Accelerate：一份训练脚本，切换所有后端

理解了 ZeRO 在切什么，下一个问题是：这些东西怎么用起来？直接用 PyTorch DDP 要 `torchrun` 启动，用 FSDP 要改模型包装，用 DeepSpeed 要写 JSON 配置加 `deepspeed` launcher——换后端就要改代码，很烦。

Accelerate 是 HuggingFace 提供的统一启动层。它**不是另一套分布式算法**，只是把后端差异藏在一个 `Accelerator` 对象后面：DDP 模式下底层是 torch.distributed，FSDP 模式下底层是 PyTorch FSDP，DeepSpeed 模式下底层就是 DeepSpeed。

In [ ]:
# === Accelerate 的统一训练循环（伪代码，展示结构） ===
code = '''
from accelerate import Accelerator

accelerator = Accelerator(
    gradient_accumulation_steps=4,   # 梯度累积：4 个 micro-batch 才算一个完整 step
    mixed_precision="bf16",          # 混合精度：工业界现在默认 bf16
)

model, optimizer, dataloader, scheduler = accelerator.prepare(
    model, optimizer, dataloader, scheduler
)

for batch in dataloader:
    with accelerator.accumulate(model):   # 梯度累积的上下文管理器
        loss = model(batch)
        accelerator.backward(loss)        # ← 取代 loss.backward()
        optimizer.step()
        optimizer.zero_grad()
'''
print(code)
print("关键观察：脚本里没有出现 DDP / FSDP / DeepSpeed 任何一个字眼。")
print("用哪个后端由启动时的配置决定，代码一行不改。")

In [ ]:
# === 启动方式：代码不动，配置动 ===
print("生成配置（交互式问答，选 DDP / FSDP / DeepSpeed + ZeRO stage）：")
print("  $ accelerate config")
print()
print("用配置文件启动：")
print("  $ accelerate launch --config_file fsdp.yaml train.py")
print()
print("也可以命令行直接指定，跳过配置文件：")
print("  $ accelerate launch --num_processes 8 --multi_gpu train.py")
print()
print("跨节点时再补 machine_rank / main_process_ip 等参数。")
print("关键观察：想从 DDP 换成 FSDP，只改配置文件，训练脚本不用动。")

三个高频参数值得单独记住：

- **`num_processes`**：GPU 总数。8 卡就是 8 个进程，每个进程跑同一份脚本、处理不同的数据分片。
- **`mixed_precision`**：现在工业界默认 `bf16`。A100/H100 原生支持 BF16，数值范围和 FP32 一样大，不需要 FP16 那套 loss scale，训练更稳。
- **`gradient_accumulation_steps`**：显存装不下大 batch 时的标配。有效 batch 的公式是：

$$\text{有效 batch} = \text{micro-batch} \times \text{GPU 数} \times \text{累积步数}$$

比如 micro-batch 4 × 8 卡 × 累积 4 步 = 有效 batch 128。单卡一次只装 4 个样本的激活，但优化 step 用的是 128 个样本的梯度。

## 4. ZeRO 的常用参数：显存不够时动哪几个旋钮

显存 OOM 是分布式训练的日常。工业界的排查顺序从代价小到代价大：

1. 先降 micro-batch、加梯度累积（零成本，用时间换空间）；
2. ZeRO Stage 2 → Stage 3；
3. `offload_optimizer`：把优化器状态卸到 CPU 内存（省最多，但 CPU↔GPU 搬运有开销）；
4. `offload_param`：参数也卸到 CPU（只有 Stage 3 支持）；
5. 实在不行再考虑 NVMe 硬盘 offload（ZeRO-Infinity，慢但能救急）。

这些旋钮在 DeepSpeed 的 JSON 配置里长这样（Accelerate 里通过 `deepspeed_config_file` 传同一份东西）：

In [ ]:
# === DeepSpeed ZeRO 配置：工业项目里最常改的几个字段 ===
import json

deepspeed_config = {
    "train_micro_batch_size_per_gpu": 4,
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,                    # 1 切优化器状态 / 2 再切梯度 / 3 再切参数
        "offload_optimizer": {         # 优化器状态卸到 CPU 内存
            "device": "cpu",
            "pin_memory": True,        # 锁页内存，CPU→GPU 拷贝更快
        },
        "offload_param": {             # 参数也卸到 CPU（仅 stage 3 生效）
            "device": "none",
        },
        "overlap_comm": True,          # 通信和计算重叠，藏掉一部分通信延迟
        "contiguous_gradients": True,  # 梯度存成连续内存块，减少通信碎片
        "reduce_bucket_size": 5e8,     # 梯度分桶大小（bytes），大桶通信高效但耗显存
    },
}

print("ds_config.json:")
print(json.dumps(deepspeed_config, indent=2))
print()
print("启动：accelerate launch --ds_config_file ds_config.json train.py")

In [ ]:
# === ZeRO 参数速查：什么时候动哪个旋钮 ===
rows = [
    ("参数", "干什么用", "什么时候动它"),
    ("stage (1/2/3)", "切优化器状态/梯度/参数", "显存不够就升级，常用 2 和 3"),
    ("offload_optimizer", "优化器状态卸到 CPU", "stage 3 还不够时，拿速度换显存"),
    ("offload_param", "参数也卸到 CPU", "模型大到 GPU 放不下，仅 stage 3"),
    ("overlap_comm", "通信与计算重叠", "默认开；多卡通信慢时确认它开着"),
    ("contiguous_gradients", "梯度连续存储", "默认开，一般不用管"),
    ("reduce_bucket_size", "梯度分桶大小", "通信是瓶颈时可调大，代价是显存"),
]
widths = [24, 28, 34]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "参数":
        print("-" * len(line))
print()
print("关键观察：真正经常改的只有 stage 和两个 offload，其余保持默认就好。")

如果后端选 FSDP 而不是 DeepSpeed，对应关系几乎一一映射：`FULL_SHARD` = ZeRO-3，`SHARD_GRAD_OP` = ZeRO-2。Accelerate 的配置文件长这样：

```yaml
compute_environment: LOCAL_MACHINE
distributed_type: FSDP
mixed_precision: bf16
num_processes: 8
fsdp_config:
  sharding_strategy: FULL_SHARD
  cpu_offload: false
  activation_checkpointing: true
```

两套后端怎么选？能力上高度重合，工程口味问题：DeepSpeed 配置驱动、offload 生态更全；FSDP 是 PyTorch 亲儿子、跟进新硬件特性最快。用 Accelerate 的好处就是可以两个都试试，代码不用改。

## 5. Megatron-LM：预训练大模型的重型武器

到目前为止的方案（ZeRO / FSDP / DeepSpeed）都是数据并行的变体。模型到几百亿、集群到几千卡时，它们会遇到天花板：ZeRO-3 每层前向都要 all-gather 参数，跨节点通信量随规模爆炸。

从零预训练大模型的工业标准是 Megatron-LM 的 **3D 并行**：

- **Tensor Parallelism（TP，张量并行）**：把每一层的矩阵乘按维度切到多卡。通信是每层都有的 all-reduce，对延迟极敏感，**只放节点内**吃 NVLink；
- **Pipeline Parallelism（PP，流水线并行）**：把模型按层切成若干段，不同段在不同节点上接力。通信量小（只传边界激活），**适合跨节点**容忍高延迟；
- **Data Parallelism（DP）**：最外层再复制几份，用 ZeRO 切冗余。

In [ ]:
# === Megatron-LM 预训练的关键参数 ===
args = [
    ("--tensor-model-parallel-size 8",  "TP 度 = 8：每层矩阵乘切 8 份，锁在节点内 8 卡"),
    ("--pipeline-model-parallel-size 16", "PP 度 = 16：模型按层切 16 段，跨 16 个节点接力"),
    ("--micro-batch-size 2",            "每卡每个 micro-step 处理 2 个样本"),
    ("--global-batch-size 1024",        "一个完整优化 step 的样本数，框架自动推算梯度累积步数"),
    ("--bf16",                          "混合精度，和 Accelerate 里的 bf16 同一件事"),
    ("--sequence-parallel",             "序列维也切分，和 TP 配合进一步省激活显存"),
    ("--recompute-activations",         "重算激活（梯度检查点），预训练省显存标配"),
]
for arg, note in args:
    print(f"{arg:<40}{note}")
print()
print("启动方式就是 torchrun --nproc_per_node 8 pretrain_gpt.py ...")
print("关键观察：TP + PP 的乘积决定「一个模型副本」占多少卡，")
print("剩下的卡全部给 DP——这就是 3D 并行的自由度分配。")

Megatron 的定位和前面完全不同：Accelerate + ZeRO 是「把你的训练脚本搬到多卡」，Megatron 是「一整套从零预训练的框架」——它连 GPT 模型定义、数据加载、loss、checkpoint 都替你写好了。NVIDIA 把并行核心抽成了 Megatron-Core 库，上层有 Megatron-LM 训练框架和 NeMo 生态；很多大厂的预训练栈都是它的变体。近年 PyTorch 官方的 torchtitan 也走 FSDP + TP 的轻量路线，适合中等规模。

一句话分工：

| 场景 | 工具 |
|:---|:---|
| 微调、几百卡以内 | Accelerate + ZeRO / FSDP |
| 从零预训练 70B+、几千卡集群 | Megatron 系（3D 并行） |
| 中小团队从零预训练 7B~13B | Accelerate + FSDP 也够用 |

（TP / PP / EP 的内部原理和手算放在附录，这一节记住「TP 锁节点内、PP 跨节点」这条工业铁律就够了。）

## 6. 微调时代的标配装备

最后补几个和分布式训练搭着用的高频开关——工业项目里几乎每个微调脚本都会出现这几个参数：

- **梯度累积**：上面讲过，显存装不下大 batch 时用时间换空间；
- **梯度检查点**（gradient checkpointing）：反向时不保存中间激活，用到时重算。激活显存省 60% 以上，整体慢约 30%；
- **BF16 混合精度**：参数和激活用 BF16 存，计算快一倍、显存省一半；
- **FlashAttention-2**：注意力计算不物化完整 attention 矩阵，长序列场景显存和速度双收益；
- **8-bit 优化器**：AdamW 状态从 12P 压到 3P bytes，效果几乎无损。

In [ ]:
# === 微调脚本里的常见参数（HuggingFace 生态写法） ===
rows = [
    ("装备", "解决什么问题", "典型参数写法"),
    ("梯度累积", "batch 大显存装不下", "gradient_accumulation_steps=8"),
    ("梯度检查点", "激活值吃显存", "gradient_checkpointing=True"),
    ("BF16 混合精度", "算得快、显存省", "bf16=True"),
    ("FlashAttention-2", "长序列注意力慢且占显存", "attn_implementation='flash_attention_2'"),
    ("8-bit 优化器", "优化器状态 12P 太大", "optim='adamw_bnb_8bit'"),
]
widths = [20, 28, 40]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "装备":
        print("-" * len(line))
print()
print("典型组合：7B 模型单卡 LoRA 微调 = bf16 + 梯度累积 + FlashAttention-2；")
print("70B 模型多卡全量微调 = bf16 + ZeRO-3 + offload + 梯度检查点。")

## 小结

- [ ] 7B 全量训练固定开销 ≈ 112 GB = 16 bytes × 参数量，DDP 多卡一点不省
- [ ] ZeRO Stage 1/2/3 分别切优化器状态、梯度、参数，越切越省、通信越多
- [ ] Accelerate 是启动层不是算法：一份代码切换 DDP / FSDP / DeepSpeed
- [ ] 有效 batch = micro-batch × GPU 数 × 梯度累积步数
- [ ] OOM 排查顺序：梯度累积 → stage 2 → stage 3 → offload optimizer → offload param
- [ ] Megatron 用 3D 并行从零预训练大模型：TP 锁节点内，PP 跨节点，DP 填满剩余
- [ ] 工业分工：微调用 Accelerate + ZeRO/FSDP，预训练大模型用 Megatron 系

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：算 ZeRO Stage 2 在 4 卡下的单卡显存**

7B 模型，AdamW 训练，4 张卡。ZeRO Stage 2 下每张卡的固定显存是多少 GB？

小提示：Stage 2 切梯度和优化器状态（14P 切成 4 份），参数仍每卡完整保留（2P）。

In [ ]:
# 作业 1：ZeRO Stage 2 在 4 卡下的单卡显存
P = 7e9
N = 4

# TODO: 计算 Stage 2 的单卡显存（单位 GB）
# 参数完整保留 + (梯度 + 优化器状态) 切 N 份
s2_per_card_gb = (2 * P + (2 * P + 12 * P) / N) / 1e9

assert s2_per_card_gb is not None, "请先计算 Stage 2 单卡显存"
expected = (2 * P + 14 * P / N) / 1e9
assert abs(s2_per_card_gb - expected) < 0.1, f"应为 {expected:.1f} GB"
print(f"✅ 作业 1 通过：")
print(f"   Stage 2 + 4 卡 + 7B：单卡 {s2_per_card_gb:.1f} GB")
print(f"   相比 DDP 的 112 GB 省了 {112 - s2_per_card_gb:.1f} GB，通信代价却几乎没涨。")

**作业 2：写一份「显存告急」的 DeepSpeed 配置**

补全下面的 `deepspeed_config`，要求：ZeRO Stage 3、优化器状态 offload 到 CPU、开启通信计算重叠。

小提示：对应 `zero_optimization.stage`、`offload_optimizer.device`、`overlap_comm` 三个字段。

In [ ]:
# 作业 2：写显存告急时的 DeepSpeed 配置
import json

deepspeed_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu"},
        "overlap_comm": True,
    },
}

# 验证
zero = deepspeed_config.get("zero_optimization", {})
assert zero.get("stage") == 3, "stage 应为 3"
assert zero.get("offload_optimizer", {}).get("device") == "cpu",     "offload_optimizer.device 应为 'cpu'"
assert zero.get("overlap_comm") is True, "overlap_comm 应为 True"

print("✅ 作业 2 通过：")
print(json.dumps(deepspeed_config, indent=2))
print()
print("这份配置 = 显存最紧张时的第一档方案：参数/梯度/优化器全切 + 优化器上 CPU。")

**作业 3：算有效 batch size**

一个预训练任务：micro-batch 2，32 张卡，梯度累积 8 步。一个完整优化 step 用了多少个样本？

小提示：三个数相乘，这就是第 3 节公式里三个因子的现实版本。

In [ ]:
# 作业 3：算有效 batch size
micro_batch = 2
num_gpus = 32
grad_accum = 8

# TODO: 计算有效 batch size
effective_batch = micro_batch * num_gpus * grad_accum

assert effective_batch is not None, "请先计算有效 batch size"
expected = micro_batch * num_gpus * grad_accum
assert effective_batch == expected, f"应为 {expected}"
print(f"✅ 作业 3 通过：")
print(f"   有效 batch = {micro_batch} × {num_gpus} × {grad_accum} = {effective_batch}")
print(f"   单卡一次只需要装 {micro_batch} 个样本的激活，却达到了 {effective_batch} 的 batch 效果。")

## 参考资料

- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020
- [HuggingFace Accelerate 文档](https://huggingface.co/docs/accelerate/)
- [DeepSpeed ZeRO 配置项文档](https://www.deepspeed.ai/docs/config-json/)
- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
- [NVIDIA Megatron-LM GitHub](https://github.com/NVIDIA/Megatron-LM)